#### Defining the configuration for data ingestion

In [0]:
ingestion_config = [
    {
        "source": "CRM",
        "path": "/Volumes/lakehouse/source/raw_data/CRM/Customer Information/",
        "folder": "Customer Information",
        "table": "cust_info"
    },
    {
        "source": "CRM",
        "path": "/Volumes/lakehouse/source/raw_data/CRM/Product Information/",
        "folder": "Product Information",
        "table": "prd_info"
    },
    {
        "source": "CRM",
        "path": "/Volumes/lakehouse/source/raw_data/CRM/Sales Details/",
        "folder": "Sales Details",
        "table": "sales_details"
    },
    {
        "source": "ERP",
        "path": "/Volumes/lakehouse/source/raw_data/ERP/Customer Location/",
        "folder": "Customer Location",
        "table": "LOC_A101"
    },
    {
        "source": "ERP",
        "path": "/Volumes/lakehouse/source/raw_data/ERP/Extra Customer Information/",
        "folder": "Extra Customer Information",
        "table": "CUST_AZ12"
    },
    {
        "source": "ERP",
        "path": "/Volumes/lakehouse/source/raw_data/ERP/Product Categories/",
        "folder": "Product Categories",
        "table": "PX_CAT_G1V2"
    }
]

#### Ingesting files into the Bronze tables

##### Creating volume inside bronze schema for checkpoints

In [0]:
spark.sql("""
          CREATE VOLUME IF NOT EXISTS lakehouse.bronze.checkpoints
          COMMENT 'Volume for checkpoint files'
          """)

In [0]:
for entity in ingestion_config:
    print(f"Ingesting {entity['source']}-{entity["folder"]} -> lakehouse.bronze.{entity['table']}")

    df_batch = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(entity["path"])
    )

    file_schema = df_batch.schema

    df = (
        spark.readStream.format("csv")
        .option("header", True)
        .schema(file_schema)
        .load(entity["path"])
    )

    (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"/Volumes/lakehouse/bronze/checkpoints/{entity['folder']}")
        .trigger(once = True)
        .toTable(f"lakehouse.bronze.{entity["table"]}")
    )